In [1]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
import geopandas as gpd

In [2]:
taz_gdf = gpd.read_file("data/USTMv4_20250729/USTMv4_20250729.shp")  # Must include CO_TAZID
tract_gdf = gpd.read_file("data/census/tl_2023_49_tract/tl_2023_49_tract.shp")  # Must include CO_TRACTID
medium_district_v3_gdf = gpd.read_file("data/USTMv3_20210922/Dist_Medium.shp")

# Get v3 to v4

In [3]:
# import global TDM functions
import sys
sys.path.insert(0, '../Resources/2-Python/global-functions')
import BigQuery

client = BigQuery.getBigQueryClient_Confidential2023UtahHTS()

bq_v3tov4 = client.query("SELECT * FROM `confidential-2023-utah-hts.geometries.ustm_v3_to_v4_best`").to_dataframe()

# check if v3_co_tazid is unique
if bq_v3tov4['v3_co_tazid'].is_unique:
    print("v3_co_tazid is unique")
else:
    print("v3_co_tazid is not unique")

# check if best_v4_co_tazid is unique
if bq_v3tov4['best_v4_co_tazid'].is_unique:
    print("best_v4_co_tazid is unique")
else:
    print("best_v4_co_tazid is not unique")
bq_v3tov4

bq_v3tov4_unique = bq_v3tov4.groupby(['best_v4_co_tazid'], as_index=False).agg(v3_co_tazid=('v3_co_tazid', 'first'))

# check if v3_co_tazid is unique
if bq_v3tov4_unique['v3_co_tazid'].is_unique:
    print("v3_co_tazid is unique")
else:
    print("v3_co_tazid is not unique")

# check if best_v4_co_tazid is unique
if bq_v3tov4_unique['best_v4_co_tazid'].is_unique:
    print("best_v4_co_tazid is unique")
else:
    print("best_v4_co_tazid is not unique")
    
bq_v3tov4_unique

bhereth
C:\Users\bhereth\confidential-2023-utah-hts-db5335615978.json


c:\Users\bhereth\AppData\Local\anaconda3\envs\july2025\Lib\site-packages\google\cloud\bigquery\table.py:1965: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


v3_co_tazid is unique
best_v4_co_tazid is not unique
v3_co_tazid is unique
best_v4_co_tazid is unique


,best_v4_co_tazid,v3_co_tazid
0,1001,1001
1,1002,1002
2,1003,1003
3,1004,1004
4,1005,1005
...,...,...
9778,570424,570424
9779,570425,570425
9780,570426,570426
9781,570427,570427


In [4]:
counties_df = pd.read_csv('results/counties.csv')
counties_df

,CO_FIPS,CO_NAME
0,1,Beaver
1,3,Box Elder
2,5,Cache
3,7,Carbon
4,9,Daggett
5,11,Davis
6,13,Duchesne
7,15,Emery
8,17,Garfield
9,19,Grand


In [5]:
prv_taz_percent_df = pd.read_csv("data/prev-v911/Lookup - BYTAZAgePct - AllCo.csv")
prv_taz_percent_df.rename(columns={';CO_TAZID':'CO_TAZID'}, inplace=True)
prv_taz_percent_df = pd.merge(prv_taz_percent_df, bq_v3tov4_unique, left_on="CO_TAZID", right_on="v3_co_tazid")
prv_taz_percent_df = prv_taz_percent_df[['best_v4_co_tazid','PCT_SUM','PCT_0TO17','PCT_18TO64','PCT_65P','SUBAREAID','CO_FIPS']]
prv_taz_percent_df.rename(columns={'best_v4_co_tazid':'CO_TAZID'}, inplace=True)
prv_taz_percent_df

,CO_TAZID,PCT_SUM,PCT_0TO17,PCT_18TO64,PCT_65P,SUBAREAID,CO_FIPS
0,1001,1,0.30,0.53,0.17,0,1
1,1002,1,0.35,0.44,0.21,0,1
2,1003,1,0.41,0.40,0.19,0,1
3,1004,1,0.30,0.53,0.17,0,1
4,1005,1,0.27,0.52,0.21,0,1
...,...,...,...,...,...,...,...
9778,570424,1,0.13,0.55,0.32,1,57
9779,570425,1,0.13,0.55,0.32,1,57
9780,570426,1,0.13,0.55,0.32,1,57
9781,570427,1,0.28,0.60,0.12,1,57


In [6]:
new_taz_percent_df = pd.read_csv("results/Lookup - BYTAZAgePct - AllCo - 2020 FOR COMPARISON.csv")
new_taz_percent_df.rename(columns={';CO_TAZID':'CO_TAZID'}, inplace=True)
new_taz_percent_df

,CO_TAZID,PCT_SUM,PCT_0TO17,PCT_18TO64,PCT_65P,SOURCE,CO_FIPS
0,1001.0,1.0,0.30,0.54,0.16,distmed,1
1,1001.0,1.0,0.30,0.51,0.19,tract,1
2,1002.0,1.0,0.34,0.44,0.22,block,1
3,1003.0,1.0,0.41,0.40,0.19,block,1
4,1004.0,1.0,0.30,0.51,0.19,tract,1
...,...,...,...,...,...,...,...
14394,570424.0,1.0,0.14,0.64,0.22,distmed,57
14395,570425.0,1.0,0.14,0.67,0.19,block,57
14396,570426.0,1.0,0.12,0.55,0.33,block,57
14397,570427.0,1.0,0.18,0.53,0.29,block,57


In [7]:
prv_taz_percent_df

,CO_TAZID,PCT_SUM,PCT_0TO17,PCT_18TO64,PCT_65P,SUBAREAID,CO_FIPS
0,1001,1,0.30,0.53,0.17,0,1
1,1002,1,0.35,0.44,0.21,0,1
2,1003,1,0.41,0.40,0.19,0,1
3,1004,1,0.30,0.53,0.17,0,1
4,1005,1,0.27,0.52,0.21,0,1
...,...,...,...,...,...,...,...
9778,570424,1,0.13,0.55,0.32,1,57
9779,570425,1,0.13,0.55,0.32,1,57
9780,570426,1,0.13,0.55,0.32,1,57
9781,570427,1,0.28,0.60,0.12,1,57


# Charts

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd

# --- NEW: source dropdown (drives filtering on the SOURCE column) ---
source_dropdown = widgets.Dropdown(
    options=[('Block', 'block'), ('Tract', 'tract'), ('Medium District', 'distmed')],
    value='tract',
    description='Source:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# Dropdown for selecting county
county_dropdown = widgets.Dropdown(
    options=[(row['CO_NAME'], row['CO_FIPS']) for _, row in counties_df.iterrows()],
    description='Select County:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# Fields to plot
fields = [
    ('% Age 0-17', 'PCT_0TO17'),
    ('% Age 18-64', 'PCT_18TO64'),
    ('% Age 65+', 'PCT_65P')
]

# Plot update function (now includes 'source')
def update_charts(co_fips, source):
    county_name = counties_df.loc[counties_df['CO_FIPS'] == co_fips, 'CO_NAME'].values[0]

    # Filter by county + SOURCE in the CURRENT dataframes
    prv_filtered = prv_taz_percent_df[
        (prv_taz_percent_df['CO_FIPS'] == co_fips)
    ]
    new_filtered = new_taz_percent_df[
        (new_taz_percent_df['CO_FIPS'] == co_fips) &
        (new_taz_percent_df['SOURCE'].astype(str).str.lower() == source)
    ]

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=[label for label, _ in fields],
        horizontal_spacing=0.05
    )

    for i, (label, field_name) in enumerate(fields, start=1):
        merged = pd.merge(
            prv_filtered[['CO_TAZID', field_name]],
            new_filtered[['CO_TAZID', field_name]],
            on='CO_TAZID',
            suffixes=('_prev', '_new'),
            how='inner'
        )

        fig.add_trace(
            go.Scatter(
                x=merged[f'{field_name}_prev'],
                y=merged[f'{field_name}_new'],
                mode='markers',
                name=label,
                text=merged['CO_TAZID'],
                marker=dict(size=8, opacity=0.7),
                showlegend=False
            ),
            row=1, col=i
        )

        # Reference line: y = x
        fig.add_trace(
            go.Scatter(
                x=[0, 1], y=[0, 1],
                mode='lines',
                line=dict(dash='dash', color='gray'),
                showlegend=False
            ),
            row=1, col=i
        )

        fig.update_xaxes(
            range=[0, 1.0], title_text='Previous',
            scaleanchor=f'y{i}', row=1, col=i
        )
        fig.update_yaxes(
            range=[0, 1.0], title_text='New',
            scaleratio=1, row=1, col=i
        )

    fig.update_layout(
        title_text=f"Change in Age Group % for {county_name} County — Source: {source.capitalize()}",
        height=600,
        width=1800,
        margin=dict(t=60, l=40, r=40, b=40)
    )

    fig.show()

# Display dropdowns + interactive plot (add 'source')
ui = widgets.VBox([county_dropdown, source_dropdown])
out = widgets.interactive_output(update_charts, {
    'co_fips': county_dropdown,
    'source': source_dropdown
})

display(ui, out)


Output()

In [9]:
new_filtered = new_taz_percent_df[new_taz_percent_df['CO_FIPS'] == 3]
new_filtered


,CO_TAZID,PCT_SUM,PCT_0TO17,PCT_18TO64,PCT_65P,SOURCE,CO_FIPS
211,3001.0,1.0,0.31,0.50,0.19,block,3
212,3002.0,1.0,0.33,0.53,0.14,distmed,3
213,3002.0,1.0,0.33,0.51,0.16,tract,3
214,3003.0,1.0,0.33,0.51,0.16,tract,3
215,3003.0,1.0,0.33,0.53,0.14,distmed,3
...,...,...,...,...,...,...,...
4078,30149.0,1.0,0.26,0.54,0.20,tract,3
4079,30150.0,1.0,0.32,0.54,0.14,block,3
4080,30151.0,1.0,0.53,0.38,0.09,block,3
4081,30152.0,1.0,0.31,0.57,0.12,tract,3


In [10]:
import pandas as pd
import geopandas as gpd
import folium
from folium import Choropleth
from branca.colormap import LinearColormap
import ipywidgets as widgets
from IPython.display import display, clear_output

# NEW: handle both Shapely 1.x and 2.x without breaking
try:
    # Shapely 2.x
    from shapely import union_all as _union_all
    def _geom_union(geoms):
        # union_all expects an array-like of geometries
        return _union_all(list(geoms))
except Exception:
    # Shapely 1.x fallback
    from shapely.ops import unary_union as _unary_union
    def _geom_union(geoms):
        return _unary_union(geoms)


# Reproject shapefile to WGS84 if needed
if taz_gdf.crs is None or taz_gdf.crs.to_epsg() != 4326:
    taz_gdf = taz_gdf.to_crs(epsg=4326)

# Dropdowns
county_dropdown = widgets.Dropdown(
    options=[(row['CO_NAME'], row['CO_FIPS']) for _, row in counties_df.iterrows()],
    description='County:',
    style={'description_width': 'initial'}
)

# --- NEW: source dropdown (drives filtering on the SOURCE column) ---
source_dropdown = widgets.Dropdown(
    options=[('Tract', 'tract'), ('Medium District', 'distmed')],
    value='tract',
    description='Non-Block Source:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

field_dropdown = widgets.Dropdown(
    options=[
        ('% Age 0-17', 'PCT_0TO17'),
        ('% Age 18-64', 'PCT_18TO64'),
        ('% Age 65+', 'PCT_65P')
    ],
    description='Compare Field:',
    style={'description_width': 'initial'}
)

# Function to update Folium map
def update_map(co_fips, source, field_name):
    clear_output(wait=True)  # avoid widget duplication on refresh
    
    # Get county name
    county_name = counties_df.loc[counties_df['CO_FIPS'] == co_fips, 'CO_NAME'].values[0]

    # Filter both DataFrames
    prv_filtered = prv_taz_percent_df[prv_taz_percent_df['CO_FIPS'] == co_fips]
    new_filtered = new_taz_percent_df[
        (new_taz_percent_df['CO_FIPS'] == co_fips) &
        ((new_taz_percent_df['SOURCE'] == source) | (new_taz_percent_df['SOURCE'] == 'block'))
    ]

    # Merge and calculate difference
    merged = pd.merge(
        prv_filtered[['CO_TAZID', field_name]],
        new_filtered[['CO_TAZID', field_name]],
        on='CO_TAZID',
        suffixes=('_prev', '_new')
    )
    merged['DIFF'] = merged[f'{field_name}_new'] - merged[f'{field_name}_prev']

    # Merge with geometry
    geo_merged = taz_gdf[['CO_TAZID', 'geometry']].merge(merged, on='CO_TAZID')
    geo_merged = geo_merged.to_crs(epsg=4326)  # Ensure WGS84

    # Get centroid for map center (robust to Shapely version)
    u = _geom_union(geo_merged.geometry)
    if u.is_empty:
        # fallback to bounds center if something odd happens
        minx, miny, maxx, maxy = geo_merged.total_bounds
        map_center = [(miny + maxy) / 2.0, (minx + maxx) / 2.0]
    else:
        c = u.centroid
        map_center = [c.y, c.x]

    # Create Folium map
    m = folium.Map(location=map_center, zoom_start=10, tiles='cartodbpositron')

    # Create color scale
    min_val, max_val = -0.1, 0.1
    colormap = LinearColormap(
        colors=['blue', 'white', 'red'],
        vmin=min_val, vmax=max_val,
        caption=f"Change in {field_name} (New - Prev)"
    )

    # Add GeoJson layer
    folium.GeoJson(
        geo_merged,
        name='TAZ Changes',
        style_function=lambda feature: {
            'fillColor': colormap(feature['properties']['DIFF']),
            'color': 'black',
            'weight': 0.5,
            'fillOpacity': 0.7,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=['CO_TAZID', 'DIFF'],
            aliases=['TAZ ID', 'Difference'],
            localize=True
        )
    ).add_to(m)

    colormap.add_to(m)

    display(m)

# Hook up widgets
ui = widgets.VBox([county_dropdown, source_dropdown, field_dropdown])
out = widgets.interactive_output(update_map, {
    'co_fips': county_dropdown,
    'source': source_dropdown,
    'field_name': field_dropdown
})

display(ui, out)


Output()

In [11]:
new_filtered = new_taz_percent_df[
    (new_taz_percent_df['CO_FIPS'] == 1) &
    ((new_taz_percent_df['SOURCE'] == 'distmed') | (new_taz_percent_df['SOURCE'] == 'block'))
]
new_filtered


,CO_TAZID,PCT_SUM,PCT_0TO17,PCT_18TO64,PCT_65P,SOURCE,CO_FIPS
0,1001.0,1.0,0.30,0.54,0.16,distmed,1
2,1002.0,1.0,0.34,0.44,0.22,block,1
3,1003.0,1.0,0.41,0.40,0.19,block,1
5,1004.0,1.0,0.30,0.54,0.16,distmed,1
6,1005.0,1.0,0.27,0.52,0.21,block,1
...,...,...,...,...,...,...,...
199,1114.0,1.0,0.32,0.57,0.11,distmed,1
200,1115.0,1.0,0.32,0.57,0.11,distmed,1
203,1116.0,1.0,0.32,0.57,0.11,distmed,1
204,1117.0,1.0,0.32,0.57,0.11,distmed,1


In [12]:
import geopandas as gpd
import folium
from folium.plugins import DualMap
from folium.features import GeoJsonTooltip
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Ensure WGS84 (EPSG:4326) for both layers ---
def _to_wgs84(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        # If CRS is missing, assume it's already WGS84; adjust if you know the source CRS
        return gdf.set_crs(epsg=4326, allow_override=True)
    if gdf.crs.to_epsg() != 4326:
        return gdf.to_crs(epsg=4326)
    return gdf

medium_district_v3_gdf = _to_wgs84(medium_district_v3_gdf)
tract_gdf['CO_FIPS'] = tract_gdf['COUNTYFP'].astype(int)  # Ensure CO_FIPS is string for consistency
tract_gdf = _to_wgs84(tract_gdf)

# --- County dropdown (CO_FIPS) ---
county_dropdown = widgets.Dropdown(
    options=[(row['CO_NAME'], row['CO_FIPS']) for _, row in counties_df.iterrows()],
    description='Select County:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px')
)

# --- Helper: common bounds across both layers for consistent view ---
def _union_bounds(gdf_list):
    minxs, minys, maxxs, maxys = zip(*(g.total_bounds for g in gdf_list if not g.empty))
    return [min(minxs), min(minys), max(maxxs), max(maxys)]

# --- Tooltip helper: pick a few safe fields if present ---
def _tooltip_fields(gdf, candidates=('CO_TAZID','GEOID','NAME','DIST_ID','MD_ID','CO_FIPS')):
    fields = [c for c in candidates if c in gdf.columns]
    aliases = [f'{c}:' for c in fields]
    return fields, aliases

# --- Map updater ---
def update_maps(co_fips):
    clear_output(wait=True)

    # Filter by CO_FIPS
    md_f = medium_district_v3_gdf  # just zoom to medium district
    #medium_district_v3_gdf[medium_district_v3_gdf['CO_FIPS'] == co_fips]
    tr_f = tract_gdf[tract_gdf['CO_FIPS'] == co_fips]

    # Safe fallback if one side is empty
    if md_f.empty and tr_f.empty:
        print(f'No geometries found for CO_FIPS={co_fips}.')
        display(ui)
        return

    # Build synced side-by-side maps
    dm = DualMap( # creates dm.m1 (left) and dm.m2 (right)
        location=[40.76, -111.89],  # placeholder; we'll fit bounds below
        tiles='cartodbpositron',
        zoom_start=10
    )

    # Left: Medium District
    md_fields, md_aliases = _tooltip_fields(md_f)
    folium.GeoJson(
        md_f,
        name='Medium District',
        style_function=lambda f: {
            'color': '#1f77b4',
            'weight': 1.0,
            'fillOpacity': 0.15
        },
        tooltip=GeoJsonTooltip(fields=md_fields, aliases=md_aliases, sticky=False) if md_fields else None,
        highlight_function=lambda f: {'weight': 2, 'fillOpacity': 0.25}
    ).add_to(dm.m1)
    folium.LayerControl(position='topright', collapsed=True).add_to(dm.m1)

    # Right: Tract
    tr_fields, tr_aliases = _tooltip_fields(tr_f)
    folium.GeoJson(
        tr_f,
        name='Tract',
        style_function=lambda f: {
            'color': '#d62728',
            'weight': 0.8,
            'fillOpacity': 0.10
        },
        tooltip=GeoJsonTooltip(fields=tr_fields, aliases=tr_aliases, sticky=False) if tr_fields else None,
        highlight_function=lambda f: {'weight': 2, 'fillOpacity': 0.2}
    ).add_to(dm.m2)
    folium.LayerControl(position='topright', collapsed=True).add_to(dm.m2)

    # Fit maps to tract bounds if not empty, else fallback to county bounds
    if not tr_f.empty:
        bounds = tr_f.total_bounds  # minx, miny, maxx, maxy
    else:
        bounds = tract_gdf[tract_gdf['CO_FIPS'] == co_fips].total_bounds

    # folium expects [southwest, northeast] [[lat, lon], [lat, lon]]
    sw = [bounds[1], bounds[0]]
    ne = [bounds[3], bounds[2]]
    dm.m1.fit_bounds([sw, ne])
    dm.m2.fit_bounds([sw, ne])

    # Titles above each pane (simple HTML markers)
    folium.map.Marker(
        location=[ne[0], (sw[1]+ne[1])/2],
        icon=folium.DivIcon(html="<div style='font-weight:600;background:rgba(255,255,255,0.8);padding:4px 8px;border-radius:6px;'>Medium District</div>")
    ).add_to(dm.m1)

    folium.map.Marker(
        location=[ne[0], (sw[1]+ne[1])/2],
        icon=folium.DivIcon(html="<div style='font-weight:600;background:rgba(255,255,255,0.8);padding:4px 8px;border-radius:6px;'>Tract</div>")
    ).add_to(dm.m2)

    # Re-show UI and map
    display(dm)

# Wire up widgets
ui = widgets.VBox([county_dropdown])
out = widgets.interactive_output(update_maps, {'co_fips': county_dropdown})

display(ui, out)


Output()